# LESSON 4.1: Fourier Transform Basics
## Filtering in the Frequency Domain

In this lesson:
- What is the Fourier Transform
- 1-D Discrete Fourier Transform (DFT)
- Sampling, aliasing, and the Nyquist rate
- The convolution theorem
- Why frequency domain filtering matters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Background: The Fourier Transform

**Jean Baptiste Joseph Fourier (1768-1830)** showed that any periodic function can be expressed as a sum of sines and cosines of different frequencies.

### Key Idea:
Any function can be decomposed into sinusoidal components, each with a specific **frequency**, **amplitude**, and **phase**.

### Why is this important for image processing?
- Low frequencies correspond to **slowly varying** intensity regions (smooth areas)
- High frequencies correspond to **rapidly changing** intensity regions (edges, noise)
- By modifying specific frequencies, we can **filter** images (blur, sharpen, denoise)

In [ ]:
# Demonstrate Fourier's idea: sum of sinusoids
t = np.linspace(0, 2*np.pi, 500)

# Individual sinusoidal components
f1 = 1.0 * np.sin(1 * t)       # frequency = 1
f2 = 0.5 * np.sin(3 * t)       # frequency = 3
f3 = 0.3 * np.sin(5 * t)       # frequency = 5
f4 = 0.2 * np.sin(7 * t)       # frequency = 7

# Sum of all components
f_sum = f1 + f2 + f3 + f4

fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True)

axes[0].plot(t, f1, 'b-')
axes[0].set_title('$1.0 \\cdot \\sin(t)$  (frequency = 1)', fontsize=11)
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, f2, 'g-')
axes[1].set_title('$0.5 \\cdot \\sin(3t)$  (frequency = 3)', fontsize=11)
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

axes[2].plot(t, f3, 'r-')
axes[2].set_title('$0.3 \\cdot \\sin(5t)$  (frequency = 5)', fontsize=11)
axes[2].set_ylabel('Amplitude')
axes[2].grid(True, alpha=0.3)

axes[3].plot(t, f4, 'm-')
axes[3].set_title('$0.2 \\cdot \\sin(7t)$  (frequency = 7)', fontsize=11)
axes[3].set_ylabel('Amplitude')
axes[3].grid(True, alpha=0.3)

axes[4].plot(t, f_sum, 'k-', linewidth=2)
axes[4].set_title('Sum of all components (complex signal)', fontsize=11)
axes[4].set_xlabel('t')
axes[4].set_ylabel('Amplitude')
axes[4].grid(True, alpha=0.3)

plt.suptitle("Fourier's Idea: A Complex Function as a Sum of Sinusoids", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. The Continuous Fourier Transform

The **Fourier Transform** of a continuous function $f(t)$ is defined as:

$$F(\mu) = \int_{-\infty}^{\infty} f(t) \, e^{-j2\pi\mu t} \, dt$$

The **Inverse Fourier Transform** recovers $f(t)$:

$$f(t) = \int_{-\infty}^{\infty} F(\mu) \, e^{j2\pi\mu t} \, d\mu$$

Where:
- $t$ = spatial (or time) variable
- $\mu$ = frequency variable
- $j = \sqrt{-1}$ (imaginary unit)
- $e^{j\theta} = \cos\theta + j\sin\theta$ (Euler's formula)

### Important: No information is lost!
The Fourier Transform is **reversible** - we can go from spatial domain to frequency domain and back without any information loss.

## 3. The 1-D Discrete Fourier Transform (DFT)

In digital processing, we work with **sampled (discrete)** data. The 1-D DFT is:

$$F(u) = \sum_{x=0}^{M-1} f(x) \, e^{-j2\pi u x / M} \quad \text{for } u = 0, 1, 2, \ldots, M-1$$

The **Inverse DFT (IDFT)** is:

$$f(x) = \frac{1}{M} \sum_{u=0}^{M-1} F(u) \, e^{j2\pi u x / M} \quad \text{for } x = 0, 1, 2, \ldots, M-1$$

Where:
- $M$ = number of samples
- $x$ = spatial variable (discrete)
- $u$ = frequency variable (discrete)
- $F(u)$ is **complex** in general

In [ ]:
# Manual DFT computation for a small signal
# f(x) = [1, 2, 4, 4]
f = np.array([1, 2, 4, 4], dtype=np.float64)
M = len(f)

# Compute DFT manually
F_manual = np.zeros(M, dtype=np.complex128)
for u in range(M):
    for x in range(M):
        F_manual[u] += f[x] * np.exp(-1j * 2 * np.pi * u * x / M)

# Compute DFT using numpy
F_numpy = np.fft.fft(f)

print("Input signal f(x):")
print(f"  f = {f}")
print()
print("DFT (Manual computation):")
for u in range(M):
    print(f"  F({u}) = {F_manual[u]:.4f}")
print()
print("DFT (NumPy FFT):")
for u in range(M):
    print(f"  F({u}) = {F_numpy[u]:.4f}")
print()
print("F(0) is the sum of all samples:", np.sum(f), "=", F_manual[0].real)

In [ ]:
# Verify inverse DFT recovers the original signal
f_recovered = np.fft.ifft(F_numpy)

print("Original signal:  ", f)
print("Recovered signal: ", np.real(f_recovered))
print("\nPerfect reconstruction: No information lost!")

## 4. Spectrum and Phase

Since $F(u)$ is complex, we can express it in polar form:

$$F(u) = |F(u)| \, e^{j\phi(u)}$$

Where:
- **Fourier Spectrum (Magnitude)**: $|F(u)| = \sqrt{R^2(u) + I^2(u)}$
- **Phase Angle**: $\phi(u) = \arctan\left(\frac{I(u)}{R(u)}\right)$
- **Power Spectrum**: $P(u) = |F(u)|^2 = R^2(u) + I^2(u)$

The **spectrum** tells us *how much* of each frequency is present.
The **phase** tells us *where* the sinusoidal components are positioned.

In [ ]:
# Create a 1-D signal and compute its spectrum
M = 256
x = np.arange(M)

# Signal: combination of sinusoids + noise
signal = (3 * np.sin(2 * np.pi * 5 * x / M) +   # frequency = 5
          1.5 * np.sin(2 * np.pi * 20 * x / M) + # frequency = 20
          0.8 * np.sin(2 * np.pi * 50 * x / M))  # frequency = 50

# Compute DFT
F = np.fft.fft(signal)
magnitude = np.abs(F)
phase = np.angle(F)

fig, axes = plt.subplots(3, 1, figsize=(12, 8))

axes[0].plot(x, signal, 'b-')
axes[0].set_title('Original Signal (spatial domain)', fontsize=12)
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].grid(True, alpha=0.3)

# Show only positive frequencies (first half)
freqs = np.arange(M//2)
axes[1].stem(freqs, magnitude[:M//2], linefmt='r-', markerfmt='ro', basefmt='k-')
axes[1].set_title('Fourier Spectrum |F(u)| (frequency domain)', fontsize=12)
axes[1].set_xlabel('Frequency (u)')
axes[1].set_ylabel('|F(u)|')
axes[1].set_xlim([0, 60])
axes[1].grid(True, alpha=0.3)

axes[2].stem(freqs, phase[:M//2], linefmt='g-', markerfmt='go', basefmt='k-')
axes[2].set_title('Phase Angle (frequency domain)', fontsize=12)
axes[2].set_xlabel('Frequency (u)')
axes[2].set_ylabel('Phase (radians)')
axes[2].set_xlim([0, 60])
axes[2].grid(True, alpha=0.3)

plt.suptitle('1-D Signal and Its Fourier Transform', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Peaks in spectrum at frequencies: 5, 20, 50 (as expected!)")

## 5. Sampling and Aliasing

### The Sampling Theorem (Nyquist-Shannon)

A continuous, band-limited function can be recovered **completely** from its samples if:

$$\frac{1}{\Delta T} > 2 \mu_{\max}$$

Where:
- $\frac{1}{\Delta T}$ = sampling rate
- $\mu_{\max}$ = highest frequency in the signal
- $2\mu_{\max}$ = **Nyquist rate** (minimum required sampling rate)

### Aliasing
When sampling rate is **below** the Nyquist rate:
- Different signals become **indistinguishable** after sampling
- Spurious frequencies appear that were **not in the original signal**
- This is called **aliasing** (one signal masquerades as another)

In [ ]:
# Demonstrate aliasing
t_fine = np.linspace(0, 1, 1000)  # "continuous" signal

# Original signal: sin(2*pi*5*t) -> frequency = 5 Hz
freq = 5  # Hz
signal_continuous = np.sin(2 * np.pi * freq * t_fine)

# Case 1: Proper sampling (above Nyquist rate)
# Nyquist rate = 2 * 5 = 10 Hz, we sample at 20 Hz
fs_good = 20
t_good = np.arange(0, 1, 1/fs_good)
samples_good = np.sin(2 * np.pi * freq * t_good)

# Case 2: Under-sampling (below Nyquist rate)
# We sample at 6 Hz (below 10 Hz Nyquist rate)
fs_bad = 6
t_bad = np.arange(0, 1, 1/fs_bad)
samples_bad = np.sin(2 * np.pi * freq * t_bad)

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# Good sampling
axes[0].plot(t_fine, signal_continuous, 'b-', alpha=0.5, label='Original (5 Hz)')
axes[0].stem(t_good, samples_good, linefmt='r-', markerfmt='ro', basefmt='k-', label=f'Sampled at {fs_good} Hz')
axes[0].set_title(f'Proper Sampling: fs = {fs_good} Hz > Nyquist rate ({2*freq} Hz)', fontsize=12)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bad sampling (aliasing)
axes[1].plot(t_fine, signal_continuous, 'b-', alpha=0.5, label='Original (5 Hz)')
axes[1].stem(t_bad, samples_bad, linefmt='r-', markerfmt='ro', basefmt='k-', label=f'Sampled at {fs_bad} Hz')
# Show aliased signal
alias_freq = fs_bad - freq  # aliased frequency = 1 Hz
axes[1].plot(t_fine, np.sin(2 * np.pi * alias_freq * t_fine), 'g--', linewidth=2, label=f'Aliased signal ({alias_freq} Hz)')
axes[1].set_title(f'Under-Sampling (ALIASING): fs = {fs_bad} Hz < Nyquist rate ({2*freq} Hz)', fontsize=12)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Amplitude')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Sampling Theorem and Aliasing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. The Convolution Theorem

One of the most important results in signal processing:

$$\boxed{(f \star h)(t) \Leftrightarrow (F \cdot H)(\mu)}$$

**Convolution in the spatial domain = Multiplication in the frequency domain**

And conversely:

$$(f \cdot h)(t) \Leftrightarrow (F \star H)(\mu)$$

**Multiplication in the spatial domain = Convolution in the frequency domain**

### Why does this matter?
- Spatial filtering (Chapter 3) uses convolution with a kernel
- The same result can be obtained by **multiplying** in the frequency domain
- For large kernels, frequency domain filtering can be **hundreds of times faster**

In [ ]:
# Demonstrate the convolution theorem
N = 256

# Create two 1-D signals
f = np.zeros(N)
f[80:180] = 1.0  # rectangular pulse

h = np.exp(-np.linspace(-3, 3, 30)**2)  # Gaussian kernel
h = h / h.sum()  # normalize
h_padded = np.zeros(N)
h_padded[:len(h)] = h

# Method 1: Spatial domain convolution
result_spatial = np.convolve(f, h, mode='same')

# Method 2: Frequency domain multiplication
F = np.fft.fft(f)
H = np.fft.fft(h_padded)
result_freq = np.real(np.fft.ifft(F * H))
# Shift to align with spatial result
result_freq = np.roll(result_freq, -len(h)//2)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(f, 'b-')
axes[0, 0].set_title('Signal f(x)', fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(h, 'r-')
axes[0, 1].set_title('Kernel h(x) (Gaussian)', fontsize=11)
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(result_spatial, 'g-', linewidth=2, label='Spatial convolution')
axes[1, 0].set_title('Result: Spatial Convolution (f * h)', fontsize=11)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

axes[1, 1].plot(result_freq, 'm-', linewidth=2, label='Freq. domain multiply')
axes[1, 1].set_title('Result: IDFT(F $\\cdot$ H)', fontsize=11)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.suptitle('Convolution Theorem: Convolution in Space = Multiplication in Frequency',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Computational Advantage of FFT

Filtering an $M \times N$ image with an $m \times n$ kernel:
- **Spatial domain**: $O(M \cdot N \cdot m \cdot n)$ operations
- **Frequency domain (FFT)**: $O(2 \cdot M \cdot N \cdot \log_2(M \cdot N))$ operations

### Computational advantage ratio:
$$C_n(m) = \frac{m^2}{4 \log_2 M}$$

For a 2048x2048 image, FFT becomes faster for kernels larger than ~7x7.

In [ ]:
# Computational advantage of FFT vs spatial filtering
M = 2048
m_values = np.arange(3, 202, 2)

# Non-separable kernel advantage
Cn = m_values**2 / (4 * np.log2(M))

# Separable kernel advantage
Cs = m_values / (2 * np.log2(M))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(m_values, Cn, 'b-', linewidth=2)
axes[0].axhline(y=1, color='r', linestyle='--', label='Break-even (C=1)')
axes[0].set_title(f'FFT vs Non-Separable Kernel (M={M})', fontsize=12)
axes[0].set_xlabel('Kernel size (m)')
axes[0].set_ylabel('Computational Advantage C(m)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

axes[1].plot(m_values, Cs, 'g-', linewidth=2)
axes[1].axhline(y=1, color='r', linestyle='--', label='Break-even (C=1)')
axes[1].set_title(f'FFT vs Separable Kernel (M={M})', fontsize=12)
axes[1].set_xlabel('Kernel size (m)')
axes[1].set_ylabel('Computational Advantage C(m)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('When FFT is Faster: C(m) > 1 favors FFT', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"For M={M}:")
print(f"  FFT advantage for m=7:   C = {7**2 / (4*np.log2(M)):.1f}x")
print(f"  FFT advantage for m=101: C = {101**2 / (4*np.log2(M)):.1f}x")
print(f"  FFT advantage for m=201: C = {201**2 / (4*np.log2(M)):.1f}x")

## Summary

What we learned:
1. **Fourier Transform** decomposes signals into sinusoidal components of different frequencies
2. **DFT** is the discrete version used in digital processing: $F(u) = \sum f(x) e^{-j2\pi ux/M}$
3. **Spectrum** $|F(u)|$ shows how much of each frequency is present
4. **Sampling theorem**: sampling rate must exceed $2\mu_{\max}$ to avoid aliasing
5. **Aliasing** introduces false frequencies when under-sampling
6. **Convolution theorem**: convolution in space = multiplication in frequency
7. **FFT** makes frequency domain filtering practical and faster for large kernels